# Simplified Non-RL Architecture Audit v1

This notebook records the current simplification checkpoint for the SOXL harness. It validates that the Alpha158 supervised/cadence path is available as the primary non-RL evidence surface and records which legacy RL compatibility surfaces remain before deletion can be considered.

In [1]:
from pathlib import Path
import json
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "soxl_quant").exists() and (candidate / "runtime").exists():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")

ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

EXPERIMENT = "simplified_non_rl_architecture_audit_v1"
CARD_DIR = ROOT / "runtime" / "harness" / "card" / EXPERIMENT
SUMMARY_PATH = CARD_DIR / "summary.json"

from src.soxl_quant.app.simplified_architecture import (
    simplified_architecture_failures,
    simplified_architecture_manifest,
)

manifest = simplified_architecture_manifest(ROOT)
failures = simplified_architecture_failures(manifest)

print("primary_surface:", manifest["primary_surface"])
print("primary_non_rl_surface_available:", manifest["primary_non_rl_surface_available"])
print("goal_complete:", manifest["goal_complete"])
print("failures:", failures)
print("remaining_rl_compatibility_surfaces:", len(manifest["remaining_rl_compatibility_surfaces"]))

primary_surface: qlib_alpha158_supervised_cadence_v1
primary_non_rl_surface_available: True
goal_complete: False
failures: []
remaining_rl_compatibility_surfaces: 2


In [2]:
audit_summary = {
    "experiment": EXPERIMENT,
    "direction": "harness",
    "decision": (
        "primary_non_rl_surface_available_legacy_rl_compatibility_remaining"
        if not failures and manifest["primary_non_rl_surface_available"] and not manifest["goal_complete"]
        else "simplified_architecture_check_failed"
    ),
    "manifest": manifest,
    "failures": failures,
    "artifacts": {
        "notebook": str((CARD_DIR / f"{EXPERIMENT}.ipynb").relative_to(ROOT)),
        "summary_json": str(SUMMARY_PATH.relative_to(ROOT)),
        "primary_model_selection_summary": manifest["primary_artifact"],
        "primary_supervised_summary": manifest["supervised_summary"],
    },
    "next_action": manifest["next_action"],
}

SUMMARY_PATH.write_text(json.dumps(audit_summary, indent=2, sort_keys=True), encoding="utf-8")
print(SUMMARY_PATH.relative_to(ROOT))

runtime/harness/card/simplified_non_rl_architecture_audit_v1/summary.json


In [3]:
selected = {
    "feature_packs": manifest["data"]["feature_packs"],
    "feature_count": manifest["data"]["feature_count"],
    "rows": manifest["data"]["rows"],
    "test_window": manifest["data"]["test_window"],
    "cadence_pair_rows": manifest["dynamic_trade_cadence"]["cadence_pair_rows"],
    "test_2022_trade_reduction_rows": manifest["dynamic_trade_cadence"]["test_2022_trade_reduction_rows"],
    "test_2022_turnover_reduction_rows": manifest["dynamic_trade_cadence"]["test_2022_turnover_reduction_rows"],
    "source_boundary_violations": manifest["source_boundary"]["violations"],
    "default_entrypoints": manifest["default_entrypoints"],
    "remaining_rl_compatibility_surfaces": manifest["remaining_rl_compatibility_surfaces"],
}
print(json.dumps(selected, indent=2, sort_keys=True))

{
  "cadence_pair_rows": 20,
  "default_entrypoints": {
    "app_module_main": "src.soxl_quant.app.simplified_cli",
    "app_package_main": "src.soxl_quant.app.simplified_cli",
    "checked_paths": [
      "soxl_qlib_strategy_v2.py",
      "src/soxl_quant/__init__.py",
      "src/soxl_quant/app/__init__.py",
      "src/soxl_quant/app/__main__.py"
    ],
    "forbidden_patterns": [
      "app.cli",
      "from .cli",
      "run_profile",
      "PPO",
      "reinforcement learning"
    ],
    "legacy_rl_cli_primary": false,
    "package_main": "src.soxl_quant.app.simplified_cli",
    "violations": []
  },
  "feature_count": 238,
  "feature_packs": [
    "qlib_alpha158",
    "release_signals_v1",
    "vix_term",
    "macro_signals",
    "soxx_signals"
  ],
  "remaining_rl_compatibility_surfaces": [
    "src/soxl_quant/policy/",
    "production/"
  ],
  "rows": 3838,
  "source_boundary_violations": [],
  "test_2022_trade_reduction_rows": 20,
  "test_2022_turnover_reduction_rows": 20,
  "te

## Conclusion

The primary non-RL architecture surface is available but the full simplification goal is not complete. The current primary evidence is `runtime/model_selection/card/qlib_alpha158_supervised_cadence_v1/summary.json`, backed by 238 features with `qlib_alpha158` first in the pack list and supervised models over the 2020-01-01 to 2026-06-15 test window. Dynamic cadence is enabled: 20 cadence rows reduced both 2022 trades and 2022 turnover.

The simplified source boundary has no forbidden `qlib.rl` or `soxl_quant.policy` imports across the checked supervised path. The default package and app-module entrypoints dispatch to `src.soxl_quant.app.simplified_cli`, not the legacy trainer CLI. The retired top-level facades and app trainer surfaces are deleted, and the remaining legacy RL compatibility surfaces are now limited to `src/soxl_quant/policy/` and `production/`; deletion should wait until runtime dependencies are migrated or explicitly retired.